# H121 vs legacy ASCAT O-F diagnostics

This notebook builds paper-style diagnostics from the monthly and full-period O-F summary files in `data/omf_compare_sums`.

The first-pass figure set emphasizes combined observation families:

- SMAP Tb species
- legacy ASCAT Metop-A/B/C species
- H SAF H121 ASCAT Metop-A/B/C species

The lower sections split the same metrics by individual species/platform so the combined diagnostics can be checked against the underlying behavior.

## Setup

In [ ]:
from __future__ import annotations

import calendar
import os
import struct
import sys
from pathlib import Path

import matplotlib as mpl
if "ipykernel" not in sys.modules:
    mpl.use("Agg")
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LogNorm, TwoSlopeNorm
from netCDF4 import Dataset

try:
    import cartopy.crs as ccrs
except Exception:
    ccrs = None

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parents[1]
elif not (ROOT / "data" / "omf_compare_sums").exists():
    ROOT = Path("/Users/amfox/Desktop/geosldas-analysis")

DATA_ROOT = ROOT / "data" / "omf_compare_sums"
OUT_DIR = ROOT / "projects" / "ascat_da" / "output" / "omf_h121_legacy_figures"
TILECOORD = ROOT / "projects" / "obs_scaling_params" / "test_data" / "inputs" / "OLv7_M36_MULTI_type_13_H121.ldas_tilecoord.bin"
OUT_DIR.mkdir(parents=True, exist_ok=True)

NMIN = 20
DPI = 180
ROLLING_WINDOW = 5

RUN_LABELS = {
    "OL_vs_SMAPobs": "OL vs SMAP obs",
    "OL_vs_legacyobs": "OL vs legacy obs",
    "OL_vs_H121obs": "OL vs H121 obs",
    "DA_H121": "H121 DA",
    "DA_legacy": "legacy DA",
    "DA_SMAP": "SMAP DA",
}

RUN_COLORS = {
    "DA_H121": "#1f77b4",
    "DA_legacy": "#d62728",
    "DA_SMAP": "#2ca02c",
}

PRIMARY_EXPERIMENTS = ("DA_H121", "DA_legacy")
ASSIMILATED_GROUPS_BY_EXPERIMENT = {
    "DA_H121": {"H121 ASCAT"},
    "DA_legacy": {"legacy ASCAT"},
    "DA_SMAP": {"SMAP"},
}

GROUP_COLORS = {
    "SMAP": "#ff7f0e",
    "legacy ASCAT": "#2ca02c",
    "H121 ASCAT": "#1f77b4",
}
GROUP_LINESTYLES = {
    "SMAP": "--",
    "legacy ASCAT": "-.",
    "H121 ASCAT": "-",
}

SPECIES_10 = [
    "SMAP_L1C_Tbh_A",
    "SMAP_L1C_Tbh_D",
    "SMAP_L1C_Tbv_A",
    "SMAP_L1C_Tbv_D",
    "ASCAT_META_SM",
    "ASCAT_METB_SM",
    "ASCAT_METC_SM",
    "ASCAT_HSAF_META_SM",
    "ASCAT_HSAF_METB_SM",
    "ASCAT_HSAF_METC_SM",
]

SPECIES_7 = SPECIES_10[:7]

EVALUATIONS = [
    {
        "group": "SMAP",
        "indices": (0, 1, 2, 3),
        "baseline": "OL_vs_SMAPobs",
        "experiments": ("DA_H121", "DA_legacy", "DA_SMAP"),
        "unit": "K",
    },
    {
        "group": "legacy ASCAT",
        "indices": (4, 5, 6),
        "baseline": "OL_vs_legacyobs",
        "experiments": ("DA_H121", "DA_legacy", "DA_SMAP"),
        "unit": "m3 m-3",
    },
    {
        "group": "H121 ASCAT",
        "indices": (7, 8, 9),
        "baseline": "OL_vs_H121obs",
        "experiments": ("DA_H121", "DA_legacy"),
        "unit": "m3 m-3",
    },
]

mpl.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": DPI,
        "font.size": 10,
        "axes.titlesize": 11,
        "axes.labelsize": 10,
        "legend.fontsize": 9,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

DATA_ROOT, OUT_DIR

## Loaders and helpers

In [ ]:
def _as_array(var):
    arr = np.array(var[:])
    if np.ma.isMaskedArray(arr):
        arr = arr.filled(np.nan)
    arr = arr.astype(float) if arr.dtype.kind in "fiu" else arr
    fill = getattr(var, "_FillValue", None)
    if fill is not None and arr.dtype.kind in "fiu":
        arr = np.where(arr == fill, np.nan, arr)
    return arr


def read_nc(path: Path) -> dict[str, np.ndarray]:
    with Dataset(path) as ds:
        out = {name: _as_array(var) for name, var in ds.variables.items()}
        out["dims"] = {name: len(dim) for name, dim in ds.dimensions.items()}
    return out


def yyyymm_to_timestamp(yyyymm):
    vals = np.asarray(yyyymm).astype(int)
    return pd.to_datetime([f"{v:06d}01" for v in vals], format="%Y%m%d")


def load_run(name: str) -> dict[str, dict[str, np.ndarray]]:
    base = DATA_ROOT / name
    monthly = read_nc(base / f"{name}_monthly_stats.nc4")
    temporal = read_nc(base / f"{name}_temporal_stats.nc4")
    monthly["months"] = yyyymm_to_timestamp(monthly["yyyymm"])
    return {"monthly": monthly, "temporal": temporal}


def _read_exact(fp, n_bytes: int) -> bytes:
    data = fp.read(n_bytes)
    if len(data) != n_bytes:
        raise EOFError(f"Expected {n_bytes} bytes, got {len(data)}")
    return data


def _read_record_tag(fp) -> int:
    return struct.unpack("<i", _read_exact(fp, 4))[0]


def read_tilecoord(path: Path) -> pd.DataFrame:
    int_fields = {"tile_id", "typ", "pfaf", "i_indg", "j_indg"}
    fields = [
        "tile_id",
        "typ",
        "pfaf",
        "com_lon",
        "com_lat",
        "min_lon",
        "max_lon",
        "min_lat",
        "max_lat",
        "i_indg",
        "j_indg",
        "frac_cell",
        "frac_pfaf",
        "area",
        "elev",
    ]
    out = {}
    with path.open("rb") as fp:
        tag = _read_record_tag(fp)
        if tag != 4:
            raise ValueError(f"{path} N_tile record tag is {tag}, expected 4")
        n_tile = struct.unpack("<i", _read_exact(fp, 4))[0]
        end_tag = _read_record_tag(fp)
        if end_tag != tag:
            raise ValueError(f"{path} N_tile record tags do not match")

        for field in fields:
            dtype = np.dtype("<i4") if field in int_fields else np.dtype("<f4")
            expected = n_tile * dtype.itemsize
            tag = _read_record_tag(fp)
            if tag != expected:
                raise ValueError(f"{path} field {field} tag is {tag}, expected {expected}")
            out[field] = np.frombuffer(_read_exact(fp, expected), dtype=dtype).copy()
            end_tag = _read_record_tag(fp)
            if end_tag != tag:
                raise ValueError(f"{path} field {field} record tags do not match")
    return pd.DataFrame(out)


def days_covered(months: pd.DatetimeIndex) -> int:
    return int(sum(calendar.monthrange(int(t.year), int(t.month))[1] for t in months))


def species_names_for_run(run_name: str) -> list[str]:
    n_species = runs[run_name]["monthly"]["OmF_stdv"].shape[1]
    return SPECIES_10[:n_species]


def has_species(data: dict[str, np.ndarray], indices) -> bool:
    return max(indices) < data["OmF_stdv"].shape[-1]


def weighted_group(values, n_data, indices, nmin=NMIN):
    if max(indices) >= values.shape[-1]:
        return np.full(values.shape[0], np.nan)
    idx = list(indices)
    vals = values[..., idx].astype(float)
    weights = n_data[..., idx].astype(float)
    valid = np.isfinite(vals) & np.isfinite(weights) & (weights >= nmin)
    num = np.where(valid, vals * weights, 0.0).sum(axis=-1)
    den = np.where(valid, weights, 0.0).sum(axis=-1)
    return np.divide(num, den, out=np.full(den.shape, np.nan), where=den > 0)


def grouped_count(n_data, indices):
    if max(indices) >= n_data.shape[-1]:
        return np.full(n_data.shape[0], np.nan)
    vals = n_data[..., list(indices)].astype(float)
    return np.where(np.isfinite(vals), vals, 0.0).sum(axis=-1)


def species_value(values, n_data, index, nmin=NMIN):
    if index >= values.shape[-1]:
        return np.full(values.shape[0], np.nan)
    vals = values[..., index].astype(float)
    weights = n_data[..., index].astype(float)
    return np.where(np.isfinite(vals) & np.isfinite(weights) & (weights >= nmin), vals, np.nan)


def percent_improvement(ol, da):
    ol = np.asarray(ol, dtype=float)
    da = np.asarray(da, dtype=float)
    return np.divide(100.0 * (ol - da), ol, out=np.full_like(ol, np.nan), where=np.isfinite(ol) & (np.abs(ol) > 0))


def percent_difference(exp, ol):
    exp = np.asarray(exp, dtype=float)
    ol = np.asarray(ol, dtype=float)
    return np.divide(100.0 * (exp - ol), ol, out=np.full_like(ol, np.nan), where=np.isfinite(ol) & (np.abs(ol) > 0))


def monthly_group_metric(cfg, run_name, metric="OmF_stdv"):
    data = runs[run_name]["monthly"]
    return weighted_group(data[metric], data["N_data"], cfg["indices"])


def temporal_group_metric(cfg, run_name, metric="OmF_stdv"):
    data = runs[run_name]["temporal"]
    return weighted_group(data[metric], data["N_data"], cfg["indices"])


def monthly_group_improvement(cfg, exp_run, metric="OmF_stdv"):
    ol = monthly_group_metric(cfg, cfg["baseline"], metric=metric)
    da = monthly_group_metric(cfg, exp_run, metric=metric)
    return percent_improvement(ol, da), ol, da


def temporal_group_improvement(cfg, exp_run, metric="OmF_stdv"):
    ol = temporal_group_metric(cfg, cfg["baseline"], metric=metric)
    da = temporal_group_metric(cfg, exp_run, metric=metric)
    return percent_improvement(ol, da), ol, da


def monthly_group_percent_difference(cfg, exp_run, metric="OmF_stdv"):
    ol = monthly_group_metric(cfg, cfg["baseline"], metric=metric)
    da = monthly_group_metric(cfg, exp_run, metric=metric)
    return percent_difference(da, ol), ol, da


def temporal_group_percent_difference(cfg, exp_run, metric="OmF_stdv"):
    ol = temporal_group_metric(cfg, cfg["baseline"], metric=metric)
    da = temporal_group_metric(cfg, exp_run, metric=metric)
    return percent_difference(da, ol), ol, da


def run_order_for_cfg(cfg):
    return (cfg["baseline"],) + tuple(cfg["experiments"])


def experiment_plot_order(cfg):
    preferred = ("DA_SMAP", "DA_legacy", "DA_H121")
    available = set(cfg["experiments"])
    return tuple(run for run in preferred if run in available)


def visible_run_order(cfg):
    preferred = (cfg["baseline"], "DA_SMAP", "DA_legacy", "DA_H121")
    available = set(run_order_for_cfg(cfg))
    return tuple(run for run in preferred if run in available)


def species_experiment_plot_order(row):
    preferred = ("DA_SMAP", "DA_legacy", "DA_H121")
    available = set(row["experiments"])
    return tuple(run for run in preferred if run in available)


def savefig(fig, stem: str):
    png = OUT_DIR / f"{stem}.png"
    pdf = OUT_DIR / f"{stem}.pdf"
    fig.savefig(png, bbox_inches="tight")
    fig.savefig(pdf, bbox_inches="tight")
    print(f"saved {png.relative_to(ROOT)}")
    print(f"saved {pdf.relative_to(ROOT)}")


def format_time_axis(ax):
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.xaxis.set_minor_locator(mdates.MonthLocator(interval=3))
    ax.grid(True, axis="y", color="0.86", linewidth=0.8)
    ax.grid(True, axis="x", color="0.93", linewidth=0.6)


def make_map_axis(fig, spec):
    if ccrs is None:
        return fig.add_subplot(spec)
    return fig.add_subplot(spec, projection=ccrs.Robinson())


MAP_EXTENT = (-180, 180, -60, 85)


def decorate_map_axis(ax):
    if ccrs is None:
        ax.set_xlim(MAP_EXTENT[0], MAP_EXTENT[1])
        ax.set_ylim(MAP_EXTENT[2], MAP_EXTENT[3])
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")
        ax.grid(True, color="0.9", linewidth=0.6)
    else:
        ax.set_extent(MAP_EXTENT, crs=ccrs.PlateCarree())
        ax.coastlines(linewidth=0.4, color="0.35")


def scatter_map(ax, lon, lat, values, *, cmap, norm=None, vmin=None, vmax=None, s=1.2):
    valid = np.isfinite(values) & np.isfinite(lon) & np.isfinite(lat)
    kwargs = dict(c=values[valid], cmap=cmap, norm=norm, vmin=vmin, vmax=vmax, s=s, linewidths=0, rasterized=True)
    if ccrs is None:
        sc = ax.scatter(lon[valid], lat[valid], **kwargs)
    else:
        sc = ax.scatter(lon[valid], lat[valid], transform=ccrs.PlateCarree(), **kwargs)
    decorate_map_axis(ax)
    return sc


## Read the O-F summaries

In [ ]:
run_names = ["OL_vs_SMAPobs", "OL_vs_legacyobs", "OL_vs_H121obs", "DA_H121", "DA_legacy", "DA_SMAP"]
runs = {name: load_run(name) for name in run_names}
tilecoord = read_tilecoord(TILECOORD)

n_tile = runs["DA_H121"]["temporal"]["OmF_stdv"].shape[0]
assert len(tilecoord) == n_tile, (len(tilecoord), n_tile)

months = runs["DA_H121"]["monthly"]["months"]
period_days = days_covered(months)
lon = tilecoord["com_lon"].to_numpy()
lat = tilecoord["com_lat"].to_numpy()

inventory_rows = []
for name, run in runs.items():
    monthly = run["monthly"]
    temporal = run["temporal"]
    inventory_rows.append(
        {
            "run": name,
            "label": RUN_LABELS[name],
            "monthly_shape": monthly["OmF_stdv"].shape,
            "temporal_shape": temporal["OmF_stdv"].shape,
            "start": monthly["months"][0].strftime("%Y-%m"),
            "end": monthly["months"][-1].strftime("%Y-%m"),
            "species": ", ".join(species_names_for_run(name)),
        }
    )

inventory = pd.DataFrame(inventory_rows)
inventory

## Combined-family summary table

In [ ]:
summary_rows = []
for cfg in EVALUATIONS:
    for exp in cfg["experiments"]:
        if not has_species(runs[exp]["monthly"], cfg["indices"]):
            continue
        month_imp, month_ol, month_da = monthly_group_improvement(cfg, exp)
        map_imp, map_ol, map_da = temporal_group_improvement(cfg, exp)
        month_diff, _, _ = monthly_group_percent_difference(cfg, exp)
        map_diff, _, _ = temporal_group_percent_difference(cfg, exp)
        summary_rows.append(
            {
                "group": cfg["group"],
                "experiment": exp,
                "experiment_label": RUN_LABELS[exp],
                "baseline": cfg["baseline"],
                "unit": cfg["unit"],
                "monthly_mean_ol_omf_stdv": np.nanmean(month_ol),
                "monthly_mean_da_omf_stdv": np.nanmean(month_da),
                "monthly_mean_pct_improvement": np.nanmean(month_imp),
                "monthly_mean_pct_difference_exp_minus_ol": np.nanmean(month_diff),
                "full_period_mean_ol_omf_stdv": np.nanmean(map_ol),
                "full_period_mean_da_omf_stdv": np.nanmean(map_da),
                "full_period_mean_pct_improvement": np.nanmean(map_imp),
                "full_period_median_pct_improvement": np.nanmedian(map_imp),
                "full_period_mean_pct_difference_exp_minus_ol": np.nanmean(map_diff),
                "full_period_median_pct_difference_exp_minus_ol": np.nanmedian(map_diff),
                "fraction_tiles_improved": np.nanmean(map_imp > 0),
                "n_valid_tiles": int(np.isfinite(map_imp).sum()),
            }
        )

summary = pd.DataFrame(summary_rows)
summary_path = OUT_DIR / "combined_omf_stdv_improvement_summary.csv"
summary.to_csv(summary_path, index=False)
summary_diff_path = OUT_DIR / "combined_omf_stdv_relative_difference_summary.csv"
summary.to_csv(summary_diff_path, index=False)
summary

## Fig. 2: experiment-first monthly O-F stddev relative difference

This follows the CYGNSS paper Figure 2 convention: `(experiment - OL) / OL * 100`. Negative values mean the DA experiment has a smaller monthly O-F stddev than the matching OL/background run for that observation family.

In [ ]:
fig, axes = plt.subplots(len(PRIMARY_EXPERIMENTS), 1, figsize=(13, 7.5), sharex=True, constrained_layout=True)
axes = np.atleast_1d(axes)
panel_labels = ["(a)", "(b)"]

for ax, exp, lab in zip(axes, PRIMARY_EXPERIMENTS, panel_labels):
    all_vals = []
    for cfg in EVALUATIONS:
        if not has_species(runs[exp]["monthly"], cfg["indices"]):
            continue
        pct, _, _ = monthly_group_percent_difference(cfg, exp)
        all_vals.append(pct)
        mean_diff = np.nanmean(pct)
        status = "DA" if cfg["group"] in ASSIMILATED_GROUPS_BY_EXPERIMENT[exp] else "MO"
        ax.plot(
            months,
            pct,
            color=GROUP_COLORS[cfg["group"]],
            linestyle=GROUP_LINESTYLES[cfg["group"]],
            linewidth=2.5 if status == "DA" else 2.0,
            label=f"{status}: {cfg['group']} ({mean_diff:.2f})",
        )

    finite = np.concatenate([v[np.isfinite(v)] for v in all_vals])
    ymin = min(-5.0, np.nanpercentile(finite, 2) - 2.0)
    ymax = max(2.0, np.nanpercentile(finite, 98) + 2.0)
    ax.set_ylim(ymin, ymax)
    ax.axhline(0, color="black", linestyle=":", linewidth=1.0)
    ax.set_ylabel("Difference (%)")
    ax.set_title(f"Normalized difference of StdDev of O-F residuals (({RUN_LABELS[exp]} - OL) / OL)", loc="center")
    ax.text(0.0, 1.06, lab, transform=ax.transAxes, va="top", ha="left")
    format_time_axis(ax)
    ax.legend(frameon=True, facecolor="white", edgecolor="gray", framealpha=0.85, loc="best")

axes[-1].set_xlabel("Month")
fig.suptitle("Monthly O-F stddev relative difference by experiment", y=1.02, fontsize=13)
savefig(fig, "fig02_experiment_monthly_omf_stdv_relative_difference")
plt.show()

## Fig. 3: experiment-first full-period O-F stddev relative-difference maps

This follows the CYGNSS paper Figure 3 convention: `(experiment - OL) / OL * 100`. Each row is an experiment and each column is an observation family. Negative values mean smaller full-period O-F stddev than the matching OL/background run. The currently cached summaries do not contain monthly spatial fields.

In [ ]:
map_values = []
for exp in PRIMARY_EXPERIMENTS:
    for cfg in EVALUATIONS:
        vals, _, _ = temporal_group_percent_difference(cfg, exp)
        map_values.append(vals)

finite = np.concatenate([v[np.isfinite(v)] for v in map_values])
lim = np.nanpercentile(np.abs(finite), 97.5)
lim = float(np.clip(lim, 15, 30))
norm = TwoSlopeNorm(vmin=-lim, vcenter=0, vmax=lim)

fig = plt.figure(figsize=(14, 6.8), constrained_layout=True)
gs = fig.add_gridspec(len(PRIMARY_EXPERIMENTS), len(EVALUATIONS))
panel_iter = iter("abcdef")
last_sc = None

for i, exp in enumerate(PRIMARY_EXPERIMENTS):
    for j, cfg in enumerate(EVALUATIONS):
        ax = make_map_axis(fig, gs[i, j])
        vals, _, _ = temporal_group_percent_difference(cfg, exp)
        status = "DA" if cfg["group"] in ASSIMILATED_GROUPS_BY_EXPERIMENT[exp] else "MO"
        mean_val = np.nanmean(vals)
        std_val = np.nanstd(vals)
        lab = next(panel_iter)
        last_sc = scatter_map(ax, lon, lat, vals, cmap="RdBu", norm=norm, s=1.05)
        ax.set_title(
            f"({RUN_LABELS[exp]} - OL) / OL, O-F StdDev {cfg['group']} ({status})\n"
            f"Mean: {mean_val:.2f} +/- {std_val:.2f} %",
            fontsize=10,
        )
        ax.text(0.0, 1.04, f"({lab})", transform=ax.transAxes, va="bottom", ha="left")

if last_sc is not None:
    cbar = fig.colorbar(last_sc, ax=fig.axes, orientation="horizontal", shrink=0.56, pad=0.05)
    cbar.set_label("%")

fig.suptitle("Full-period O-F stddev relative difference by experiment", y=1.03, fontsize=13)
savefig(fig, "fig03_experiment_full_period_omf_stdv_relative_difference_maps")
plt.show()

## Fig. 4: experiment-first monthly O-F stddev values

This shows the absolute monthly O-F stddev for the three experiment states: OL, H121 DA, and legacy DA. The OL panel uses each observation family's matching OL monitor run. ASCAT families are on the left axis (`m3 m-3`); SMAP Tb is on the right axis (`K`).

In [ ]:
ABSOLUTE_STDDEV_EXPERIMENTS = (
    ("OL", "OL references"),
    ("DA_H121", RUN_LABELS["DA_H121"]),
    ("DA_legacy", RUN_LABELS["DA_legacy"]),
)


def monthly_omf_stdv_for_experiment(cfg, exp_key):
    run_name = cfg["baseline"] if exp_key == "OL" else exp_key
    return monthly_group_metric(cfg, run_name, metric="OmF_stdv")


left_values = []
right_values = []
for exp_key, _ in ABSOLUTE_STDDEV_EXPERIMENTS:
    for cfg in EVALUATIONS:
        vals = monthly_omf_stdv_for_experiment(cfg, exp_key)
        if cfg["group"] == "SMAP":
            right_values.append(vals)
        else:
            left_values.append(vals)

left_finite = np.concatenate([v[np.isfinite(v)] for v in left_values])
right_finite = np.concatenate([v[np.isfinite(v)] for v in right_values])
left_ylim = (0.0, float(np.nanpercentile(left_finite, 99) * 1.12))
right_ylim = (0.0, float(np.nanpercentile(right_finite, 99) * 1.12))

fig, axes = plt.subplots(len(ABSOLUTE_STDDEV_EXPERIMENTS), 1, figsize=(13, 9.4), sharex=True, constrained_layout=True)
axes = np.atleast_1d(axes)
panel_labels = ["(a)", "(b)", "(c)"]

for ax, (exp_key, exp_label), lab in zip(axes, ABSOLUTE_STDDEV_EXPERIMENTS, panel_labels):
    ax_smap = ax.twinx()
    handles = []
    labels = []

    for cfg in EVALUATIONS:
        vals = monthly_omf_stdv_for_experiment(cfg, exp_key)
        target_ax = ax_smap if cfg["group"] == "SMAP" else ax
        if exp_key == "OL":
            status = "OL"
        else:
            status = "DA" if cfg["group"] in ASSIMILATED_GROUPS_BY_EXPERIMENT[exp_key] else "MO"
        mean_val = np.nanmean(vals)
        unit_label = "K" if cfg["group"] == "SMAP" else "m3 m-3"
        line = target_ax.plot(
            months,
            vals,
            color=GROUP_COLORS[cfg["group"]],
            linestyle=GROUP_LINESTYLES[cfg["group"]],
            linewidth=2.6 if status == "DA" else 2.1,
            label=f"{status}: {cfg['group']} ({mean_val:.3g} {unit_label})",
        )[0]
        handles.append(line)
        labels.append(line.get_label())

    ax.set_ylim(*left_ylim)
    ax_smap.set_ylim(*right_ylim)
    ax.set_ylabel("ASCAT O-F stddev\n(m3 m-3)")
    ax_smap.set_ylabel("SMAP O-F stddev\n(K)")
    ax.set_title(f"Monthly O-F stddev: {exp_label}", loc="center")
    ax.text(0.0, 1.06, lab, transform=ax.transAxes, va="top", ha="left")
    format_time_axis(ax)
    ax_smap.grid(False)
    ax.legend(handles, labels, frameon=True, facecolor="white", edgecolor="gray", framealpha=0.85, loc="best")

axes[-1].set_xlabel("Month")
fig.suptitle("Monthly O-F stddev by experiment", y=1.02, fontsize=13)
savefig(fig, "fig04_experiment_monthly_omf_stdv_values_dual_axis")
plt.show()

## Support A: observation support by family

These panels show the observation density in the matching OL/background files. This helps interpret whether apparent skill differences are happening on comparable spatial support.

In [ ]:
density = {}
for cfg in EVALUATIONS:
    data = runs[cfg["baseline"]]["temporal"]
    density[cfg["group"]] = grouped_count(data["N_data"], cfg["indices"]) / period_days

positive = np.concatenate([v[np.isfinite(v) & (v > 0)] for v in density.values()])
vmin = max(np.nanpercentile(positive, 2), 1e-3)
vmax = np.nanpercentile(positive, 98)
norm_count = LogNorm(vmin=vmin, vmax=vmax)

fig = plt.figure(figsize=(12, 4.3), constrained_layout=True)
gs = fig.add_gridspec(1, len(EVALUATIONS))
last_sc = None
for i, cfg in enumerate(EVALUATIONS):
    ax = make_map_axis(fig, gs[0, i])
    vals = density[cfg["group"]]
    last_sc = scatter_map(ax, lon, lat, vals, cmap="viridis", norm=norm_count, s=1.1)
    ax.set_title(f"{cfg['group']}\nmedian={np.nanmedian(vals):.2g} obs/day")

if last_sc is not None:
    cbar = fig.colorbar(last_sc, ax=fig.axes, orientation="horizontal", shrink=0.72, pad=0.04)
    cbar.set_label("Observations per tile per day")

fig.suptitle("Full-period observation density by family", y=1.03, fontsize=13)
savefig(fig, "supportA_combined_observation_density_maps")
plt.show()

## Support B: monthly observation counts by family

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.0), constrained_layout=True)
family_colors = {"SMAP": "#4c78a8", "legacy ASCAT": "#f58518", "H121 ASCAT": "#54a24b"}

for cfg in EVALUATIONS:
    data = runs[cfg["baseline"]]["monthly"]
    counts = grouped_count(data["N_data"], cfg["indices"])
    ax.plot(months, counts / 1e6, marker="o", ms=3.0, linewidth=1.8, color=family_colors[cfg["group"]], label=cfg["group"])

ax.set_ylabel("Monthly observations (million)")
ax.set_xlabel("Month")
ax.set_title("Monthly observation counts in matching OL/background files", loc="left")
format_time_axis(ax)
ax.legend(ncol=3, frameon=False)
savefig(fig, "supportB_combined_monthly_observation_counts")
plt.show()

## Species-level helper table

This table assigns each species to the baseline run that contains the corresponding observations.

In [ ]:
species_rows = []
for i, name in enumerate(SPECIES_10):
    if i < 4:
        group = "SMAP"
        baseline = "OL_vs_SMAPobs"
        experiments = ("DA_H121", "DA_legacy", "DA_SMAP")
    elif i < 7:
        group = "legacy ASCAT"
        baseline = "OL_vs_legacyobs"
        experiments = ("DA_H121", "DA_legacy", "DA_SMAP")
    else:
        group = "H121 ASCAT"
        baseline = "OL_vs_H121obs"
        experiments = ("DA_H121", "DA_legacy")
    species_rows.append({"index": i, "species": name, "group": group, "baseline": baseline, "experiments": experiments})

species_table = pd.DataFrame(species_rows)
species_table

## Fig. 5: species-level monthly O-F stddev improvement

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(13, 12), sharex=True, constrained_layout=True)
axes = axes.ravel()

for ax, row in zip(axes, species_rows):
    idx = row["index"]
    base = runs[row["baseline"]]["monthly"]
    ol = species_value(base["OmF_stdv"], base["N_data"], idx)
    for exp in species_experiment_plot_order(row):
        exp_data = runs[exp]["monthly"]
        if idx >= exp_data["OmF_stdv"].shape[-1]:
            continue
        da = species_value(exp_data["OmF_stdv"], exp_data["N_data"], idx)
        imp = percent_improvement(ol, da)
        smooth = pd.Series(imp, index=months).rolling(ROLLING_WINDOW, center=True, min_periods=1).mean()
        zorder = 5 if exp == "DA_H121" else 2
        marker = "o" if exp == "DA_H121" else None
        ax.plot(months, smooth, color=RUN_COLORS[exp], linewidth=2.0 if exp == "DA_H121" else 1.7, marker=marker, markersize=2.4 if exp == "DA_H121" else 0, markevery=6 if exp == "DA_H121" else None, zorder=zorder, label=RUN_LABELS[exp])
        ax.scatter(months, imp, s=10 if exp == "DA_H121" else 8, color=RUN_COLORS[exp], alpha=0.28 if exp == "DA_H121" else 0.18, zorder=zorder)
    ax.axhline(0, color="0.25", linewidth=0.7)
    ax.set_title(f"{row['species']} ({row['group']})", loc="left", fontsize=10)
    format_time_axis(ax)

for ax in axes[::2]:
    ax.set_ylabel("Improvement (%)")
for ax in axes[-2:]:
    ax.set_xlabel("Month")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 1.02))
fig.suptitle("Species-level monthly O-F stddev improvement", y=1.045, fontsize=13)
savefig(fig, "fig05_species_monthly_omf_stdv_improvement")
plt.show()

## Fig. 6: species-level full-period O-F stddev improvement bars

In [ ]:
bar_rows = []
for row in species_rows:
    idx = row["index"]
    base = runs[row["baseline"]]["temporal"]
    ol = species_value(base["OmF_stdv"], base["N_data"], idx)
    for exp in row["experiments"]:
        exp_data = runs[exp]["temporal"]
        if idx >= exp_data["OmF_stdv"].shape[-1]:
            continue
        da = species_value(exp_data["OmF_stdv"], exp_data["N_data"], idx)
        imp = percent_improvement(ol, da)
        bar_rows.append(
            {
                "species": row["species"],
                "group": row["group"],
                "experiment": exp,
                "experiment_label": RUN_LABELS[exp],
                "mean_pct_improvement": np.nanmean(imp),
                "median_pct_improvement": np.nanmedian(imp),
                "fraction_tiles_improved": np.nanmean(imp > 0),
                "n_valid_tiles": int(np.isfinite(imp).sum()),
            }
        )

species_summary = pd.DataFrame(bar_rows)
species_summary_path = OUT_DIR / "species_omf_stdv_improvement_summary.csv"
species_summary.to_csv(species_summary_path, index=False)

fig, ax = plt.subplots(figsize=(13, 5.8), constrained_layout=True)
species_order = [row["species"] for row in species_rows]
x = np.arange(len(species_order))
width = 0.24
offsets = {"DA_H121": -width, "DA_legacy": 0.0, "DA_SMAP": width}

for exp in ("DA_H121", "DA_legacy", "DA_SMAP"):
    vals = []
    for species in species_order:
        hit = species_summary[(species_summary["species"] == species) & (species_summary["experiment"] == exp)]
        vals.append(np.nan if hit.empty else float(hit["mean_pct_improvement"].iloc[0]))
    ax.bar(x + offsets[exp], vals, width=width, color=RUN_COLORS[exp], label=RUN_LABELS[exp])

ax.axhline(0, color="0.25", linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(species_order, rotation=35, ha="right")
ax.set_ylabel("Mean tile improvement (%)")
ax.set_title("Full-period O-F stddev improvement by species", loc="left")
ax.grid(True, axis="y", color="0.88", linewidth=0.8)
ax.legend(ncol=3, frameon=False)
savefig(fig, "fig06_species_full_period_omf_stdv_improvement_bars")
plt.show()

species_summary

## All combined-family metrics

This section writes long-form monthly values and full-period tile summaries for every metric available in the summary files, grouped into SMAP, legacy ASCAT, and H121 ASCAT families.

In [ ]:
MONTHLY_METRICS = [
    "O_mean",
    "O_stdv",
    "F_mean",
    "F_stdv",
    "A_mean",
    "A_stdv",
    "OmF_mean",
    "OmF_stdv",
    "OmA_mean",
    "OmA_stdv",
    "N_data",
]
TEMPORAL_METRICS = MONTHLY_METRICS + ["OmF_norm_mean", "OmF_norm_stdv"]
MEAN_METRICS = ["O_mean", "F_mean", "A_mean", "OmF_mean", "OmA_mean"]
STDV_METRICS = ["O_stdv", "F_stdv", "A_stdv", "OmF_stdv", "OmA_stdv"]

METRIC_LABELS = {
    "O_mean": "O mean",
    "O_stdv": "O stddev",
    "F_mean": "F mean",
    "F_stdv": "F stddev",
    "A_mean": "A mean",
    "A_stdv": "A stddev",
    "OmF_mean": "O-F mean",
    "OmF_stdv": "O-F stddev",
    "OmA_mean": "O-A mean",
    "OmA_stdv": "O-A stddev",
    "OmF_norm_mean": "normalized O-F mean",
    "OmF_norm_stdv": "normalized O-F stddev",
    "N_data": "observation count",
}


def grouped_metric(data, cfg, metric):
    if metric not in data or not has_species(data, cfg["indices"]):
        return None
    if metric == "N_data":
        return grouped_count(data["N_data"], cfg["indices"])
    return weighted_group(data[metric], data["N_data"], cfg["indices"])


combined_monthly_rows = []
for cfg in EVALUATIONS:
    for run_name in run_order_for_cfg(cfg):
        data = runs[run_name]["monthly"]
        for metric in MONTHLY_METRICS:
            vals = grouped_metric(data, cfg, metric)
            if vals is None:
                continue
            for month, value in zip(months, vals):
                combined_monthly_rows.append(
                    {
                        "month": month.strftime("%Y-%m"),
                        "group": cfg["group"],
                        "run": run_name,
                        "run_label": RUN_LABELS[run_name],
                        "baseline": cfg["baseline"],
                        "metric": metric,
                        "metric_label": METRIC_LABELS[metric],
                        "value": value,
                    }
                )

combined_monthly_metrics = pd.DataFrame(combined_monthly_rows)
combined_monthly_path = OUT_DIR / "combined_monthly_all_metrics.csv"
combined_monthly_metrics.to_csv(combined_monthly_path, index=False)

combined_temporal_rows = []
for cfg in EVALUATIONS:
    for run_name in run_order_for_cfg(cfg):
        data = runs[run_name]["temporal"]
        for metric in TEMPORAL_METRICS:
            vals = grouped_metric(data, cfg, metric)
            if vals is None:
                continue
            combined_temporal_rows.append(
                {
                    "group": cfg["group"],
                    "run": run_name,
                    "run_label": RUN_LABELS[run_name],
                    "baseline": cfg["baseline"],
                    "metric": metric,
                    "metric_label": METRIC_LABELS[metric],
                    "mean": np.nanmean(vals),
                    "median": np.nanmedian(vals),
                    "std": np.nanstd(vals),
                    "p05": np.nanpercentile(vals, 5),
                    "p95": np.nanpercentile(vals, 95),
                    "n_valid_tiles": int(np.isfinite(vals).sum()),
                }
            )

combined_temporal_metrics = pd.DataFrame(combined_temporal_rows)
combined_temporal_path = OUT_DIR / "combined_temporal_all_metrics_summary.csv"
combined_temporal_metrics.to_csv(combined_temporal_path, index=False)

combined_monthly_metrics.head(), combined_temporal_metrics.head()

## Fig. 7: all combined-family monthly mean metrics

In [ ]:
def plot_combined_metric_grid(metrics, stem, title):
    fig, axes = plt.subplots(len(metrics), len(EVALUATIONS), figsize=(15.2, 2.1 * len(metrics) + 1.0), sharex=True, constrained_layout=True)
    if len(metrics) == 1:
        axes = axes[np.newaxis, :]

    legend_handles = {}
    for row, metric in enumerate(metrics):
        for col, cfg in enumerate(EVALUATIONS):
            ax = axes[row, col]
            for run_name in visible_run_order(cfg):
                data = runs[run_name]["monthly"]
                vals = grouped_metric(data, cfg, metric)
                if vals is None or not np.isfinite(vals).any():
                    continue
                color = "0.25" if run_name == cfg["baseline"] else RUN_COLORS.get(run_name, "0.5")
                linestyle = "--" if run_name == cfg["baseline"] else "-"
                linewidth = 1.5 if run_name == cfg["baseline"] else 1.8
                marker = "o" if run_name == "DA_H121" else None
                markersize = 2.6 if run_name == "DA_H121" else 0
                markevery = 6 if run_name == "DA_H121" else None
                zorder = 5 if run_name == "DA_H121" else 2
                y = vals / 1e6 if metric == "N_data" else vals
                handle = ax.plot(
                    months,
                    y,
                    color=color,
                    linestyle=linestyle,
                    linewidth=linewidth,
                    marker=marker,
                    markersize=markersize,
                    markevery=markevery,
                    zorder=zorder,
                    label=RUN_LABELS[run_name],
                )[0]
                legend_handles[RUN_LABELS[run_name]] = handle
            ax.set_title(f"{cfg['group']} - {METRIC_LABELS[metric]}", loc="left", fontsize=10)
            ylabel = "million obs" if metric == "N_data" else "weighted family mean"
            ax.set_ylabel(ylabel)
            format_time_axis(ax)

    for ax in axes[-1, :]:
        ax.set_xlabel("Month")
    fig.legend(list(legend_handles.values()), list(legend_handles.keys()), loc="center left", ncol=1, frameon=False, bbox_to_anchor=(1.01, 0.5))
    fig.suptitle(title, y=1.015, fontsize=13)
    savefig(fig, stem)
    plt.show()


plot_combined_metric_grid(MEAN_METRICS, "fig07_combined_monthly_mean_metrics", "Combined-family monthly mean metrics")

## Fig. 8: all combined-family monthly stddev metrics

In [ ]:
plot_combined_metric_grid(STDV_METRICS, "fig08_combined_monthly_stdv_metrics", "Combined-family monthly stddev metrics")

## Notes for later tweaks

- The combined-family metrics use `N_data` as species weights and require at least `NMIN` observations for each species contribution.
- The maps are full-period aggregates because the available temporal files collapse the 72 months into one tile-level statistic.
- A DA run is always compared to the OL/background file containing the matching observation family: SMAP uses `OL_vs_SMAPobs`, legacy ASCAT uses `OL_vs_legacyobs`, and H121 ASCAT uses `OL_vs_H121obs`.
- The notebook writes PNG/PDF figure pairs and CSV summaries to `projects/ascat_da/output/omf_h121_legacy_figures`.